### Libraries Import

In [11]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

### Load Dataset

In [12]:
df = pd.read_csv(
    ROOT / "datasets/processed/cleaned_paddy.csv",
    keep_default_na=False,
    encoding="utf-8-sig"
)

In [13]:
model_df = df[
    [
        "qa_id",
        "pair_id",
        "question",
        "language",
        "category_en",
        "issue_id",
        "issue_en"
    ]
].copy()

model_df = model_df.rename(
    columns={"category_en": "label"}
)

### Filter empty questions and labels

In [14]:
model_df = model_df[
    model_df["question"].str.strip().ne("")
    & model_df["label"].str.strip().ne("")
]

label_names = sorted(
    model_df["label"].unique().tolist()
)

### Category Names

In [15]:
label_to_id = {
    label: idx
    for idx, label in enumerate(label_names)
}

### add label_id Column

In [16]:
model_df["label_id"] = (
    model_df["label"]
    .map(label_to_id)
    .astype(int)
)

### Save data set

In [17]:
# Keep only rows with usable text and labels.
model_df = model_df[
    model_df["question"].str.strip().ne("")
    & model_df["label"].str.strip().ne("")
].copy()

label_names = sorted(model_df["label"].unique().tolist())
label_to_id = {label: idx for idx, label in enumerate(label_names)}
model_df["label_id"] = model_df["label"].map(label_to_id).astype(int)

out = ROOT / "datasets/processed/model_ready_paddy.csv"
out.parent.mkdir(parents=True, exist_ok=True)
model_df.to_csv(out, index=False, encoding="utf-8-sig")

label_map_path = ROOT / "models/problem_classifier/label_map.json"
label_map_path.parent.mkdir(parents=True, exist_ok=True)
label_map_path.write_text(
    pd.Series(label_to_id).to_json(force_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Saved {len(model_df):,} rows to {out}")
print(f"Unique labels: {len(label_names)}")

Saved 25,440 rows to d:\My_Project\Analyse\AI-Paddy-Farming-Intelligence\datasets\processed\model_ready_paddy.csv
Unique labels: 12
